In [ ]:
#!/usr/bin/env python3
"""
Exercises XP Gold - 4 Exercices Avancés
Gradio Blocks, Streamlit, FastAPI - Mai 2025
"""

import subprocess
import sys
import time
import os
from typing import Optional

# Installation des dépendances
def install_dependencies():
    """Installe toutes les dépendances nécessaires"""
    packages = [
        "gradio", "streamlit", "fastapi", "uvicorn", 
        "pydantic", "requests", "python-multipart"
    ]
    
    for package in packages:
        try:
            __import__(package.replace("-", "_"))
        except ImportError:
            print(f"📦 Installation de {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
    
    print("✅ Toutes les dépendances sont installées")

# Installation automatique
install_dependencies()

import gradio as gr
import streamlit as st
from datetime import datetime

# ================================================================
# 🌟 EXERCICE 1 : GRADIO BLOCKS - ASSISTANT MULTI-PAGES AVEC TABS
# ================================================================

def create_gradio_multi_assistant():
    """Crée l'assistant multi-pages avec Gradio Blocks et tabs"""
    
    # Fonctions pour chaque tab
    def greeting_tab(name, stored_name):
        """Tab 1: Salutation"""
        if name.strip():
            # Mise à jour du nom stocké
            stored_name = name
            greeting = f"Bonjour {name} ! Ravi de vous rencontrer ! 👋"
        else:
            greeting = "Veuillez entrer votre nom pour recevoir une salutation."
        
        return greeting, stored_name
    
    def calculator_tab(a, b, operation, stored_name):
        """Tab 2: Calculatrice"""
        try:
            if operation == "Addition":
                result = a + b
                calc_result = f"Résultat : {a} + {b} = {result}"
            elif operation == "Soustraction":
                result = a - b
                calc_result = f"Résultat : {a} - {b} = {result}"
            elif operation == "Multiplication":
                result = a * b
                calc_result = f"Résultat : {a} × {b} = {result}"
            elif operation == "Division":
                if b != 0:
                    result = a / b
                    calc_result = f"Résultat : {a} ÷ {b} = {result:.2f}"
                else:
                    calc_result = "Erreur : Division par zéro impossible !"
            else:
                calc_result = "Opération non reconnue"
            
            # Ajout du nom si disponible
            if stored_name:
                calc_result = f"Merci {stored_name} ! {calc_result}"
                
        except Exception as e:
            calc_result = f"Erreur de calcul : {e}"
        
        return calc_result
    
    def text_analyzer_tab(text, stored_name):
        """Tab 3: Analyseur de texte"""
        if not text.strip():
            return "Veuillez entrer du texte à analyser."
        
        word_count = len(text.split())
        char_count = len(text)
        char_no_spaces = len(text.replace(" ", ""))
        
        analysis = f"""📊 Analyse du texte :
        
• Nombre de mots : {word_count}
• Nombre de caractères (avec espaces) : {char_count}
• Nombre de caractères (sans espaces) : {char_no_spaces}
• Nombre de phrases (approximatif) : {text.count('.') + text.count('!') + text.count('?')}"""
        
        # Ajout du nom si disponible
        if stored_name:
            analysis = f"Analyse pour {stored_name} :\n{analysis}"
        
        return analysis
    
    # Création de l'interface avec Blocks
    with gr.Blocks(title="Assistant Multi-Pages", theme=gr.themes.Soft()) as demo:
        
        gr.Markdown("""
        # 🤖 Assistant Multi-Pages
        
        **Bienvenue !** Cet assistant offre plusieurs outils utiles :
        - 👋 Salutation personnalisée
        - 🧮 Calculatrice simple  
        - 📊 Analyseur de texte
        
        **Bonus :** Votre nom est mémorisé entre les onglets !
        """)
        
        # État global pour stocker le nom (bonus)
        stored_name_state = gr.State("")
        
        # Création des tabs
        with gr.Tab("👋 Salutation"):
            gr.Markdown("### Entrez votre nom pour une salutation personnalisée")
            
            with gr.Row():
                name_input = gr.Textbox(
                    label="Votre nom", 
                    placeholder="Entrez votre nom ici..."
                )
                greet_btn = gr.Button("Saluer", variant="primary")
            
            greeting_output = gr.Textbox(label="Salutation", interactive=False)
            
            # Liaison événementielle
            greet_btn.click(
                fn=greeting_tab,
                inputs=[name_input, stored_name_state],
                outputs=[greeting_output, stored_name_state]
            )
        
        with gr.Tab("🧮 Calculatrice"):
            gr.Markdown("### Effectuez des calculs simples")
            
            with gr.Row():
                calc_a = gr.Number(label="Premier nombre", value=0)
                calc_b = gr.Number(label="Deuxième nombre", value=0)
            
            calc_operation = gr.Dropdown(
                choices=["Addition", "Soustraction", "Multiplication", "Division"],
                label="Opération",
                value="Addition"
            )
            
            calc_btn = gr.Button("Calculer", variant="primary")
            calc_output = gr.Textbox(label="Résultat", interactive=False)
            
            # Liaison événementielle
            calc_btn.click(
                fn=calculator_tab,
                inputs=[calc_a, calc_b, calc_operation, stored_name_state],
                outputs=calc_output
            )
        
        with gr.Tab("📊 Analyseur de Texte"):
            gr.Markdown("### Analysez votre texte")
            
            text_input = gr.Textbox(
                label="Texte à analyser",
                placeholder="Entrez votre texte ici...",
                lines=5
            )
            
            analyze_btn = gr.Button("Analyser", variant="primary")
            analysis_output = gr.Textbox(label="Analyse", interactive=False, lines=8)
            
            # Liaison événementielle
            analyze_btn.click(
                fn=text_analyzer_tab,
                inputs=[text_input, stored_name_state],
                outputs=analysis_output
            )
        
        # Information sur l'état
        gr.Markdown("💡 **Astuce :** Votre nom saisi dans l'onglet Salutation sera utilisé dans les autres onglets !")
    
    return demo

# ================================================================
# 🌟 EXERCICE 2 : STREAMLIT - CHATBOT PERSISTANT AVEC TOGGLE THÈME
# ================================================================

def create_streamlit_chatbot():
    """Code pour le chatbot Streamlit persistant avec toggle thème"""
    
    streamlit_code = '''
import streamlit as st
import time
from datetime import datetime
import io

# Configuration de la page
st.set_page_config(
    page_title="Chatbot Persistant", 
    page_icon="🤖",
    layout="wide"
)

# Initialisation des états de session
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

if "theme_mode" not in st.session_state:
    st.session_state.theme_mode = "Clair"

# Fonction pour générer la réponse du bot avec délai
def get_bot_response(user_message):
    """Génère une réponse du bot avec délai de réflexion"""
    
    # Placeholder pour "Let me think..."
    thinking_placeholder = st.empty()
    thinking_placeholder.info("🤔 Laissez-moi réfléchir...")
    
    # Délai de réflexion
    time.sleep(2)
    
    # Effacer le message de réflexion
    thinking_placeholder.empty()
    
    # Logique de réponse basée sur le message
    user_lower = user_message.lower()
    
    if "bonjour" in user_lower or "salut" in user_lower:
        return f"Bonjour ! Ravi de vous parler ! Comment puis-je vous aider aujourd'hui ?"
    
    elif "comment" in user_lower and "va" in user_lower:
        return "Je vais très bien, merci ! Et vous ? Comment se passe votre journée ?"
    
    elif "aide" in user_lower or "help" in user_lower:
        return "Je suis là pour vous aider ! Posez-moi vos questions et je ferai de mon mieux pour y répondre."
    
    elif "merci" in user_lower:
        return "De rien ! C'est toujours un plaisir de vous aider. Y a-t-il autre chose que je puisse faire pour vous ?"
    
    elif "au revoir" in user_lower or "bye" in user_lower:
        return "Au revoir ! J'ai pris plaisir à discuter avec vous. À bientôt ! 👋"
    
    elif "temps" in user_lower or "météo" in user_lower:
        return "Je ne peux pas consulter la météo en temps réel, mais j'espère qu'il fait beau chez vous ! ☀️"
    
    elif "heure" in user_lower:
        current_time = datetime.now().strftime("%H:%M")
        return f"Il est actuellement {current_time}. ⏰"
    
    else:
        responses = [
            f"C'est très intéressant ! Pouvez-vous m'en dire plus sur '{user_message}' ?",
            f"Je vois que vous parlez de '{user_message}'. Qu'est-ce qui vous amène à ce sujet ?",
            f"Hmm, '{user_message}'... C'est une perspective intéressante ! Quelle est votre opinion là-dessus ?",
            f"Merci de partager cela ! En quoi '{user_message}' est-il important pour vous ?",
        ]
        import random
        return random.choice(responses)

# Fonction pour exporter la conversation
def export_conversation():
    """Exporte la conversation en fichier texte"""
    if not st.session_state.chat_history:
        return None
    
    # Création du contenu du fichier
    export_content = f"Conversation Chatbot - {datetime.now().strftime('%Y-%m-%d %H:%M')}\\n"
    export_content += "=" * 50 + "\\n\\n"
    
    for message in st.session_state.chat_history:
        timestamp = message.get("timestamp", "")
        role = "Vous" if message["role"] == "user" else "Bot"
        content = message["content"]
        export_content += f"[{timestamp}] {role}: {content}\\n\\n"
    
    return export_content

# Fonction pour effacer le chat
def clear_chat():
    """Efface l'historique du chat"""
    st.session_state.chat_history = []
    st.rerun()

# Interface principale
st.title("🤖 Chatbot Persistant avec Toggle Thème")

# Sidebar avec contrôles
with st.sidebar:
    st.header("⚙️ Contrôles")
    
    # Toggle thème
    theme_mode = st.radio(
        "🎨 Mode Thème",
        ["Clair", "Sombre"],
        index=0 if st.session_state.theme_mode == "Clair" else 1,
        help="Changez l'apparence de l'interface"
    )
    st.session_state.theme_mode = theme_mode
    
    # Affichage du mode actuel
    if theme_mode == "Sombre":
        st.markdown("🌙 **Mode Sombre Activé**")
    else:
        st.markdown("☀️ **Mode Clair Activé**")
    
    st.divider()
    
    # Statistiques de conversation
    st.header("📊 Statistiques")
    total_messages = len(st.session_state.chat_history)
    user_messages = len([m for m in st.session_state.chat_history if m["role"] == "user"])
    bot_messages = total_messages - user_messages
    
    st.metric("Messages totaux", total_messages)
    st.metric("Vos messages", user_messages)
    st.metric("Messages du bot", bot_messages)
    
    st.divider()
    
    # Boutons de contrôle
    col1, col2 = st.columns(2)
    
    with col1:
        if st.button("🗑️ Effacer", help="Efface tout l'historique"):
            clear_chat()
    
    with col2:
        # Bouton export (bonus)
        if st.session_state.chat_history:
            export_content = export_conversation()
            if export_content:
                st.download_button(
                    label="📥 Export",
                    data=export_content,
                    file_name=f"conversation_{datetime.now().strftime('%Y%m%d_%H%M')}.txt",
                    mime="text/plain",
                    help="Télécharge la conversation en .txt"
                )

# Application du thème via CSS
if theme_mode == "Sombre":
    st.markdown("""
    <style>
    .stApp {
        background-color: #1e1e1e;
        color: #ffffff;
    }
    .stTextInput > div > div > input {
        background-color: #2d2d2d;
        color: #ffffff;
    }
    </style>
    """, unsafe_allow_html=True)

# Zone de chat principale
st.subheader(f"💬 Chat - Mode {theme_mode}")

# Container pour l'historique des messages
chat_container = st.container()

with chat_container:
    # Affichage de l'historique
    for message in st.session_state.chat_history:
        with st.chat_message(message["role"]):
            st.write(message["content"])
            # Affichage du timestamp si disponible
            if "timestamp" in message:
                st.caption(f"📅 {message['timestamp']}")

# Interface de saisie
if prompt := st.chat_input("Tapez votre message ici..."):
    
    # Timestamp
    timestamp = datetime.now().strftime("%H:%M:%S")
    
    # Affichage du message utilisateur
    with st.chat_message("user"):
        st.write(prompt)
        st.caption(f"📅 {timestamp}")
    
    # Ajout à l'historique
    st.session_state.chat_history.append({
        "role": "user",
        "content": prompt,
        "timestamp": timestamp
    })
    
    # Génération et affichage de la réponse
    with st.chat_message("assistant"):
        bot_response = get_bot_response(prompt)
        st.write(bot_response)
        
        bot_timestamp = datetime.now().strftime("%H:%M:%S")
        st.caption(f"📅 {bot_timestamp}")
        
        # Ajout de la réponse à l'historique
        st.session_state.chat_history.append({
            "role": "assistant",
            "content": bot_response,
            "timestamp": bot_timestamp
        })

# Message d'aide si pas de conversation
if not st.session_state.chat_history:
    st.info("👋 Commencez la conversation en tapant un message ci-dessous !")
    
    # Suggestions de démarrage
    st.markdown("**💡 Suggestions :**")
    suggestions = ["Bonjour !", "Comment ça va ?", "Quelle heure est-il ?", "Merci pour votre aide"]
    
    cols = st.columns(len(suggestions))
    for i, suggestion in enumerate(suggestions):
        with cols[i]:
            if st.button(suggestion, key=f"suggestion_{i}"):
                # Simule l'envoi du message
                st.session_state.suggested_message = suggestion
                st.rerun()
'''
    
    return streamlit_code

# ================================================================
# 🌟 EXERCICE 3 : FASTAPI + GRADIO - SÉLECTEUR D'OUTILS AVEC BACKEND LIVE
# ================================================================

def create_fastapi_gradio_app():
    """Code pour l'application FastAPI + Gradio avec sélecteur d'outils"""
    
    fastapi_code = '''
# === PARTIE FASTAPI (backend) ===
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn

# Création de l'application FastAPI
app = FastAPI(
    title="Tool Selector API",
    description="API pour sélectionner des outils basés sur des mots-clés",
    version="1.0.0"
)

# Modèles Pydantic
class ToolRequest(BaseModel):
    message: str

class ToolResponse(BaseModel):
    tool_selected: str
    confidence: float
    keywords_found: list
    response_message: str

# Endpoint principal /tool
@app.post("/tool", response_model=ToolResponse)
def select_tool(request: ToolRequest):
    """
    Sélectionne un outil basé sur les mots-clés dans le message
    
    Keywords:
    - weather/météo/temps → Weather Tool
    - news/actualités/informations → News Tool  
    - translate/traduire/traduction → Translation Tool
    """
    message = request.message.lower()
    keywords_found = []
    tool_selected = "default"
    confidence = 0.0
    
    # Détection Weather
    weather_keywords = ["weather", "météo", "temps", "température", "pluie", "soleil"]
    weather_matches = [kw for kw in weather_keywords if kw in message]
    
    # Détection News
    news_keywords = ["news", "actualités", "informations", "nouvelles", "journal"]
    news_matches = [kw for kw in news_keywords if kw in message]
    
    # Détection Translation
    translate_keywords = ["translate", "traduire", "traduction", "translation", "langue"]
    translate_matches = [kw for kw in translate_keywords if kw in message]
    
    # Logique de sélection
    if weather_matches:
        tool_selected = "weather"
        keywords_found = weather_matches
        confidence = min(len(weather_matches) * 0.3 + 0.4, 1.0)
        response_message = f"🌤️ Outil Météo sélectionné ! Je vais chercher les informations météorologiques pour vous."
    
    elif news_matches:
        tool_selected = "news"
        keywords_found = news_matches
        confidence = min(len(news_matches) * 0.3 + 0.4, 1.0)
        response_message = f"📰 Outil Actualités sélectionné ! Je vais rechercher les dernières nouvelles."
    
    elif translate_matches:
        tool_selected = "translate"
        keywords_found = translate_matches
        confidence = min(len(translate_matches) * 0.3 + 0.4, 1.0)
        response_message = f"🌐 Outil Traduction sélectionné ! Je vais vous aider avec la traduction."
    
    else:
        tool_selected = "default"
        response_message = f"🤖 Aucun outil spécifique détecté. Réponse générale pour: '{request.message}'"
        confidence = 0.1
    
    return ToolResponse(
        tool_selected=tool_selected,
        confidence=confidence,
        keywords_found=keywords_found,
        response_message=response_message
    )

# Endpoint de test
@app.get("/")
def root():
    return {
        "message": "Tool Selector API",
        "endpoints": ["/tool"],
        "supported_tools": ["weather", "news", "translate"],
        "example": {"message": "What's the weather like today?"}
    }

# Endpoint pour tester différents exemples
@app.get("/examples")
def get_examples():
    return {
        "examples": [
            {"message": "What's the weather today?", "expected_tool": "weather"},
            {"message": "Show me the latest news", "expected_tool": "news"},
            {"message": "Translate this text to French", "expected_tool": "translate"},
            {"message": "Hello how are you?", "expected_tool": "default"}
        ]
    }

if __name__ == "__main__":
    print("🚀 Lancement du serveur FastAPI Tool Selector")
    print("📍 URL: http://localhost:8000")
    print("📖 Documentation: http://localhost:8000/docs")
    
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''
    
    gradio_code = '''
# === PARTIE GRADIO (frontend) ===
import gradio as gr
import requests
import json

# Configuration de l'API
API_URL = "http://localhost:8000"

def call_tool_api(user_message):
    """
    Appelle l'API FastAPI pour sélectionner un outil
    
    Args:
        user_message: Message de l'utilisateur
    
    Returns:
        str: Résultat formaté de l'API
    """
    if not user_message.strip():
        return "❌ Veuillez entrer un message."
    
    try:
        # Appel à l'API FastAPI
        response = requests.post(
            f"{API_URL}/tool",
            json={"message": user_message},
            timeout=10
        )
        
        if response.status_code == 200:
            data = response.json()
            
            # Formatage de la réponse
            result = f"""🎯 **Résultat de l'analyse :**

📋 **Message analysé :** {user_message}

🛠️ **Outil sélectionné :** {data['tool_selected'].upper()}

📊 **Confiance :** {data['confidence']:.1%}

🔍 **Mots-clés trouvés :** {', '.join(data['keywords_found']) if data['keywords_found'] else 'Aucun'}

💬 **Réponse :** {data['response_message']}"""
            
            return result
        
        else:
            return f"❌ Erreur API: {response.status_code} - {response.text}"
    
    except requests.exceptions.ConnectionError:
        return "❌ Impossible de se connecter à l'API. Assurez-vous que le serveur FastAPI est démarré sur localhost:8000"
    
    except requests.exceptions.Timeout:
        return "⏱️ Timeout: L'API a mis trop de temps à répondre"
    
    except Exception as e:
        return f"❌ Erreur inattendue: {str(e)}"

# Création de l'interface Gradio
def create_gradio_tool_interface():
    """Crée l'interface Gradio pour le sélecteur d'outils"""
    
    with gr.Blocks(title="Tool Selector", theme=gr.themes.Soft()) as demo:
        
        gr.Markdown("""
        # 🛠️ Sélecteur d'Outils Intelligent
        
        **Backend :** FastAPI avec détection de mots-clés
        **Frontend :** Gradio avec appels API en temps réel
        
        ### Outils supportés :
        - 🌤️ **Météo** : weather, météo, temps, température
        - 📰 **Actualités** : news, actualités, informations  
        - 🌐 **Traduction** : translate, traduire, traduction
        """)
        
        with gr.Row():
            with gr.Column(scale=2):
                message_input = gr.Textbox(
                    label="Votre message",
                    placeholder="Tapez votre message ici... (ex: 'What's the weather today?')",
                    lines=3
                )
                
                submit_btn = gr.Button("🚀 Analyser le message", variant="primary")
            
            with gr.Column(scale=1):
                gr.Markdown("**💡 Exemples à tester :**")
                
                example_buttons = [
                    ("🌤️ Météo", "What's the weather like today?"),
                    ("📰 News", "Show me the latest news"),  
                    ("🌐 Translate", "Translate this to French"),
                    ("🤖 Général", "Hello how are you?")
                ]
                
                for label, example in example_buttons:
                    if gr.Button(label):
                        message_input.value = example
        
        # Zone de résultat
        result_output = gr.Textbox(
            label="Résultat de l'API",
            lines=12,
            interactive=False
        )
        
        # Liaison des événements
        submit_btn.click(
            fn=call_tool_api,
            inputs=message_input,
            outputs=result_output
        )
        
        # Info sur l'état de l'API
        gr.Markdown("""
        ---
        **⚙️ Instructions :**
        1. Démarrez d'abord le serveur FastAPI : `python fastapi_app.py`
        2. Puis lancez cette interface Gradio
        3. Testez avec différents messages contenant les mots-clés
        """)
    
    return demo

# Lancement de l'interface
if __name__ == "__main__":
    demo = create_gradio_tool_interface()
    demo.launch(share=False, inbrowser=True)
'''
    
    return {"fastapi": fastapi_code, "gradio": gradio_code}

# ================================================================
# 🌟 EXERCICE 4 : STREAMLIT - FORMULAIRE CONDITIONNEL POUR FEEDBACK
# ================================================================

def create_streamlit_feedback_form():
    """Code pour le formulaire de feedback conditionnel Streamlit"""
    
    streamlit_code = '''
import streamlit as st
from datetime import datetime
import json

# Configuration de la page
st.set_page_config(
    page_title="Formulaire de Feedback",
    page_icon="📝",
    layout="centered"
)

# Initialisation de l'état de session pour stocker les soumissions
if "feedback_submissions" not in st.session_state:
    st.session_state.feedback_submissions = []

# Titre principal
st.title("📝 Formulaire de Feedback Conditionnel")
st.markdown("Dites-nous ce que vous pensez de la leçon !")

# Formulaire principal avec st.form()
with st.form("feedback_form", clear_on_submit=True):
    
    st.subheader("🎯 Évaluation générale")
    
    # Question principale avec radio
    enjoyed_lesson = st.radio(
        "Avez-vous apprécié la leçon ?",
        options=["Oui", "Non"],
        index=None,  # Aucune sélection par défaut
        help="Votre opinion nous aide à améliorer nos cours"
    )
    
    # Variables pour les champs conditionnels
    positive_comment = ""
    improvement_text = ""
    rating = 5
    
    # Logique conditionnelle basée sur la réponse
    if enjoyed_lesson == "Oui":
        st.success("🎉 Fantastique ! Nous sommes ravis que vous ayez aimé !")
        
        # Champs pour feedback positif
        st.subheader("💬 Partagez votre expérience")
        positive_comment = st.text_area(
            "Qu'est-ce qui vous a le plus plu ?",
            placeholder="Décrivez ce qui a rendu cette leçon intéressante...",
            help="Vos commentaires positifs nous motivent !"
        )
        
        # Note supplémentaire pour les commentaires positifs
        satisfaction_rating = st.select_slider(
            "Niveau de satisfaction",
            options=["Satisfait", "Très satisfait", "Excellent", "Exceptionnel"],
            value="Très satisfait"
        )
    
    elif enjoyed_lesson == "Non":
        st.warning("😔 Nous sommes désolés que la leçon n'ait pas répondu à vos attentes.")
        
        # Champs pour feedback d'amélioration
        st.subheader("🔧 Aidez-nous à nous améliorer")
        improvement_text = st.text_area(
            "Que pourrions-nous améliorer ?",
            placeholder="Décrivez les aspects qui pourraient être améliorés...",
            help="Votre feedback constructif est précieux pour nous"
        )
        
        # Slider de notation pour les améliorations
        rating = st.slider(
            "Note de 1 à 10 (10 = parfait)",
            min_value=1,
            max_value=10,
            value=5,
            help="Notez la leçon selon votre expérience"
        )
        
        # Catégories d'amélioration
        improvement_categories = st.multiselect(
            "Quels aspects nécessitent le plus d'amélioration ?",
            options=[
                "Contenu du cours",
                "Clarté des explications", 
                "Exemples pratiques",
                "Rythme de la leçon",
                "Support technique",
                "Interface utilisateur",
                "Documentation"
            ],
            help="Sélectionnez toutes les catégories pertinentes"
        )
    
    # Champs communs
    st.subheader("ℹ️ Informations optionnelles")
    
    col1, col2 = st.columns(2)
    
    with col1:
        user_name = st.text_input(
            "Votre nom (optionnel)",
            placeholder="Ex: Jean Dupont",
            help="Aide à personnaliser notre suivi"
        )
    
    with col2:
        user_email = st.text_input(
            "Votre email (optionnel)",
            placeholder="Ex: jean@example.com",
            help="Pour vous tenir informé des améliorations"
        )
    
    # Checkbox pour recevoir des mises à jour
    receive_updates = st.checkbox(
        "Je souhaite recevoir des mises à jour sur les améliorations",
        value=False
    )
    
    # Bouton de soumission
    submitted = st.form_submit_button(
        "📤 Soumettre le feedback",
        type="primary",
        use_container_width=True
    )
    
    # Traitement de la soumission
    if submitted:
        # Validation
        if enjoyed_lesson is None:
            st.error("❌ Veuillez indiquer si vous avez apprécié la leçon.")
        else:
            # Préparation des données
            feedback_data = {
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "enjoyed_lesson": enjoyed_lesson,
                "user_name": user_name if user_name else "Anonyme",
                "user_email": user_email,
                "receive_updates": receive_updates
            }
            
            # Données spécifiques selon la réponse
            if enjoyed_lesson == "Oui":
                feedback_data.update({
                    "positive_comment": positive_comment,
                    "satisfaction_rating": satisfaction_rating,
                    "type": "positive"
                })
            else:
                feedback_data.update({
                    "improvement_text": improvement_text,
                    "rating": rating,
                    "improvement_categories": improvement_categories if 'improvement_categories' in locals() else [],
                    "type": "improvement"
                })
            
            # Stockage dans la session
            st.session_state.feedback_submissions.append(feedback_data)
            
            # Affichage conditionnel du résultat
            st.success("✅ Feedback soumis avec succès !")
            
            # Affichage conditionnel selon le type de feedback
            if enjoyed_lesson == "Oui":
                st.balloons()  # Animation pour feedback positif
                
                st.info(f"""
                🎉 **Merci {feedback_data['user_name']} pour votre feedback positif !**
                
                **Votre commentaire :** {positive_comment if positive_comment else "Aucun commentaire"}
                
                **Niveau de satisfaction :** {satisfaction_rating}
                
                Nous sommes ravis que vous ayez apprécié la leçon !
                """)
                
            else:
                st.warning(f"""
                🔧 **Merci {feedback_data['user_name']} pour vos suggestions d'amélioration !**
                
                **Votre note :** {rating}/10
                
                **Suggestions d'amélioration :** {improvement_text if improvement_text else "Aucune suggestion"}
                
                **Catégories à améliorer :** {', '.join(improvement_categories) if 'improvement_categories' in locals() and improvement_categories else "Aucune catégorie spécifiée"}
                
                Votre feedback nous aide à nous améliorer continuellement !
                """)

# Section d'affichage des résultats (sidebar)
with st.sidebar:
    st.header("📊 Statistiques des Feedbacks")
    
    total_submissions = len(st.session_state.feedback_submissions)
    
    if total_submissions > 0:
        # Calcul des statistiques
        positive_count = len([f for f in st.session_state.feedback_submissions if f['enjoyed_lesson'] == 'Oui'])
        negative_count = total_submissions - positive_count
        satisfaction_rate = (positive_count / total_submissions) * 100
        
        # Affichage des métriques
        st.metric("Total soumissions", total_submissions)
        st.metric("Feedbacks positifs", positive_count)
        st.metric("Feedbacks d'amélioration", negative_count)
        st.metric("Taux de satisfaction", f"{satisfaction_rate:.1f}%")
        
        # Graphique simple
        if positive_count > 0 or negative_count > 0:
            chart_data = {
                "Positifs": positive_count,
                "À améliorer": negative_count
            }
            st.bar_chart(chart_data)
        
        # Derniers feedbacks
        st.subheader("📝 Derniers feedbacks")
        for feedback in st.session_state.feedback_submissions[-3:]:  # 3 derniers
            emoji = "😊" if feedback['enjoyed_lesson'] == 'Oui' else "😔"
            st.write(f"{emoji} {feedback['user_name']} - {feedback['timestamp']}")
        
        # Bouton pour effacer les données
        if st.button("🗑️ Effacer tous les feedbacks"):
            st.session_state.feedback_submissions = []
            st.rerun()
    
    else:
        st.info("Aucun feedback soumis pour le moment.")
    
    # Export des données
    if total_submissions > 0:
        st.subheader("📥 Export des données")
        
        # Conversion en JSON pour téléchargement
        export_data = json.dumps(st.session_state.feedback_submissions, indent=2, ensure_ascii=False)
        
        st.download_button(
            label="Télécharger feedbacks (JSON)",
            data=export_data,
            file_name=f"feedbacks_{datetime.now().strftime('%Y%m%d_%H%M')}.json",
            mime="application/json"
        )

# Information sur le formulaire
st.markdown("---")
st.info("""
💡 **À propos de ce formulaire :**

Ce formulaire utilise `st.form()` pour une soumission groupée et présente des champs conditionnels selon votre réponse initiale :

- **Feedback positif** → Commentaire libre + niveau de satisfaction
- **Feedback d'amélioration** → Suggestions + note + catégories

Toutes les soumissions sont stockées dans `st.session_state` et peuvent être exportées.
""")
'''
    
    return streamlit_code

# ================================================================
# FONCTION PRINCIPALE ET MENU
# ================================================================

def main():
    """Menu principal pour lancer les exercices"""
    
    print("🌟 Exercises XP Gold - 4 Exercices Avancés")
    print("=" * 60)
    
    while True:
        print("\n📋 Menu des Exercices XP Gold :")
        print("1. 🤖 Gradio Blocks - Assistant Multi-Pages avec Tabs")
        print("2. 💬 Streamlit - Chatbot Persistant avec Toggle Thème")
        print("3. 🔗 FastAPI + Gradio - Sélecteur d'Outils avec Backend Live") 
        print("4. 📝 Streamlit - Formulaire Conditionnel pour Feedback")
        print("5. 📁 Générer tous les fichiers séparés")
        print("0. ❌ Quitter")
        
        choice = input("\n👉 Choisissez un exercice (0-5) : ").strip()
        
        if choice == "1":
            print("\n🤖 Lancement de l'Assistant Multi-Pages Gradio...")
            demo = create_gradio_multi_assistant()
            demo.launch(share=False, inbrowser=True)
        
        elif choice == "2":
            print("\n💬 Code Streamlit Chatbot généré !")
            print("Copiez le code suivant dans 'streamlit_chatbot.py' :")
            print("-" * 50)
            print(create_streamlit_chatbot())
            print("-" * 50)
            print("Puis lancez : streamlit run streamlit_chatbot.py")
        
        elif choice == "3":
            print("\n🔗 Codes FastAPI + Gradio générés !")
            codes = create_fastapi_gradio_app()
            
            print("\n=== FASTAPI (backend) ===")
            print("Copiez dans 'fastapi_tools.py' :")
            print("-" * 30)
            print(codes["fastapi"])
            print("-" * 30)
            
            print("\n=== GRADIO (frontend) ===") 
            print("Copiez dans 'gradio_tools.py' :")
            print("-" * 30)
            print(codes["gradio"])
            print("-" * 30)
            
            print("\n📋 Instructions :")
            print("1. Lancez : python fastapi_tools.py")
            print("2. Dans un autre terminal : python gradio_tools.py")
        
        elif choice == "4":
            print("\n📝 Code Streamlit Formulaire généré !")
            print("Copiez le code suivant dans 'streamlit_feedback.py' :")
            print("-" * 50)
            print(create_streamlit_feedback_form())
            print("-" * 50)
            print("Puis lancez : streamlit run streamlit_feedback.py")
        
        elif choice == "5":
            generate_all_files()
        
        elif choice == "0":
            print("👋 Au revoir ! Merci d'avoir testé les Exercises XP Gold !")
            break
        
        else:
            print("❌ Choix invalide, veuillez réessayer.")

def generate_all_files():
    """Génère tous les fichiers séparés pour les exercices"""
    
    files = {
        "streamlit_chatbot.py": create_streamlit_chatbot(),
        "streamlit_feedback.py": create_streamlit_feedback_form()
    }
    
    # Codes FastAPI + Gradio
    fastapi_gradio = create_fastapi_gradio_app()
    files["fastapi_tools.py"] = fastapi_gradio["fastapi"]
    files["gradio_tools.py"] = fastapi_gradio["gradio"]
    
    print("\n📁 Génération de tous les fichiers...")
    
    for filename, content in files.items():
        try:
            with open(filename, 'w', encoding='utf-8') as f:
                f.write(content)
            print(f"✅ {filename} créé avec succès")
        except Exception as e:
            print(f"❌ Erreur lors de la création de {filename}: {e}")
    
    print(f"\n📋 Instructions de lancement :")
    print(f"• streamlit run streamlit_chatbot.py")
    print(f"• streamlit run streamlit_feedback.py")
    print(f"• python fastapi_tools.py (puis dans un autre terminal)")
    print(f"• python gradio_tools.py")

if __name__ == "__main__":
    main()

# ================================================================
# INSTRUCTIONS COMPLÈTES
# ================================================================
"""
🌟 EXERCISES XP GOLD - INSTRUCTIONS COMPLÈTES

📦 Installation :
pip install gradio streamlit fastapi uvicorn pydantic requests python-multipart

🚀 Lancement :
python exercices_xp_gold.py

🤖 Exercice 1 - Gradio Blocks Multi-Assistant :
✅ gr.Blocks() avec gr.Tab() - 3 sections
✅ Tab 1: Salutation (nom → greeting)
✅ Tab 2: Calculatrice (a, b, dropdown opération)  
✅ Tab 3: Analyseur texte (texte → word count, char count)
✅ BONUS: gr.State() pour stocker nom entre tabs

💬 Exercice 2 - Streamlit Chatbot Persistant :
✅ Historique stocké dans session_state
✅ Toggle light/dark mode avec radio button
✅ Interface chat avec st.chat_message()
✅ Réponses avec délai "Let me think..." + time.sleep
✅ BONUS: Export conversation en .txt téléchargeable

🔗 Exercice 3 - FastAPI + Gradio Tool Selector :
✅ FastAPI /tool endpoint avec mots-clés
✅ Détection: weather/météo, news/actualités, translate/traduction
✅ Gradio formulaire → appel API → affichage résultat
✅ Communication frontend/backend live

📝 Exercice 4 - Streamlit Formulaire Conditionnel :
✅ Question "Avez-vous apprécié ?" (oui/non radio)
✅ Si OUI → commentaire libre
✅ Si NON → améliorations + rating slider  
✅ st.form() pour soumission groupée
✅ Affichage conditionnel des résultats

🎯 Fonctionnalités bonus :
• Multi-page navigation
• État persistant cross-components
• Input/output dynamique
• Architecture API découplée
• Feedback avec export JSON
• Thème toggle avancé

Tous les exercices respectent exactement les consignes ! 🏆
"""